<a href="https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This turns the validated Week-5/6 model into something a human can actually act on. Three
pieces: a probability-ranked queue with interpretable reason codes (not just a number), a light
K-Means archetype layer in the spirit of FlyRank's own "Content Archetypes" appendix (descriptive
segments, not production personas), and a decay/refresh insight grounded in *this* dataset's own
`is_declining_label` rather than borrowed from the paper.

The model here is refit on the full portfolio (not just the training split) so the queue covers
every page -- but the number I trust it by is the Week-6 held-out, client-grouped result
(precision@20 = 0.75, ROC AUC = 0.586, base rate 0.511), not anything computed on this full-data
fit. A full-data fit can score more pages; it cannot certify itself.

In [ ]:
import os, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

DATA = "data/raw/content_refresh_anonymized.csv"
if not Path(DATA).exists():
    if not Path("flyrank-ml-internship").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Eng7ouda06/flyrank-ml-internship.git"], check=True)
    os.chdir("flyrank-ml-internship")

df = pd.read_csv(DATA)
df["is_declining_label"] = (df.trend_direction == "down").astype(int)

# same honest feature set as Weeks 5-6
numeric_features = ["impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d", "users_90d",
                    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
                    "days_with_impressions", "days_with_sessions", "content_age_days",
                    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
                    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
                    "word_count", "char_count"]
categorical_features = ["content_type", "main_intent", "competition_level"]

def build_X(frame):
    X_num = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    cols_with_na = X_num.columns[X_num.isna().any()]
    missing_flags = X_num[cols_with_na].isna().astype(int).add_suffix("_missing")
    X_num = X_num.fillna(0)
    X_cat = pd.get_dummies(frame[categorical_features].fillna("unknown").astype(str), dummy_na=False)
    return pd.concat([X_num.reset_index(drop=True), missing_flags.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)

X = build_X(df)
y = df["is_declining_label"].astype(int)

# validated numbers from Week 6 (client-grouped holdout) -- the credibility statement for this queue
VALIDATED = {"precision@20": 0.75, "precision@50": 0.80, "precision@100": 0.74, "roc_auc": 0.586,
             "avg_precision": 0.589, "base_rate": 0.511, "split": "client-grouped, 7 held-out clients"}
BASELINE_VALIDATED = {"precision@20": 0.50, "precision@50": 0.56, "precision@100": 0.54, "roc_auc": 0.520}

full_model = Pipeline([("scaler", StandardScaler()),
                        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])
full_model.fit(X, y)
df["decline_probability"] = full_model.predict_proba(X)[:, 1]

# --- archetypes: light K-Means, descriptive only ---
cluster_features = ["impressions_90d", "avg_position", "content_age_days", "days_since_last_update",
                     "word_count", "ai_traffic_pct"]
Xc = df[cluster_features].apply(pd.to_numeric, errors="coerce").fillna(0).copy()
Xc["impressions_90d"] = np.log1p(Xc["impressions_90d"])
Xc_scaled = StandardScaler().fit_transform(Xc)
km = KMeans(n_clusters=4, n_init=10, random_state=42)
df["archetype_id"] = km.fit_predict(Xc_scaled)

archetype_profile = df.groupby("archetype_id").agg(
    n=("content_id", "count"), avg_impressions=("impressions_90d", "mean"), avg_position=("avg_position", "mean"),
    avg_age=("content_age_days", "mean"), avg_freshness=("days_since_last_update", "mean"),
    avg_words=("word_count", "mean"), decline_rate=("is_declining_label", "mean")).round(1)
print(archetype_profile)

SMALL_CLUSTER_ID = int(archetype_profile["n"].idxmin())
archetype_names = {}
for aid, row in archetype_profile.iterrows():
    if aid == SMALL_CLUSTER_ID:
        archetype_names[aid] = "small_edge_case_cluster"
    elif row["avg_freshness"] > 60 and row["avg_impressions"] > archetype_profile["avg_impressions"].median():
        archetype_names[aid] = "high_visibility_stale"
    elif row["avg_age"] < archetype_profile["avg_age"].median():
        archetype_names[aid] = "young_high_volume"
    else:
        archetype_names[aid] = "established_recently_updated"
df["archetype"] = df["archetype_id"].map(archetype_names)
print(archetype_names)

                  n  avg_impressions  avg_position  avg_age  avg_freshness  \
archetype_id                                                                 
0              8590           3700.4          18.7    418.7           22.1   
1             12169           4566.1          13.7    143.7           17.9   
2              9147           7498.8          17.6    253.2          106.3   
3                94            726.2          19.1    261.0           34.3   

              avg_words  decline_rate  
archetype_id                           
0                1939.4           0.4  
1                3087.5           0.6  
2                3946.7           0.6  
3                2868.8           0.5  
{0: 'established_recently_updated', 1: 'young_high_volume', 2: 'high_visibility_stale', 3: 'small_edge_case_cluster'}


**Reason codes** are interpretable rules layered on top of the model probability -- someone
reviewing the queue should never have to trust a bare number. **Suggested action** combines the
reason codes with the archetype: `small_edge_case_cluster` and anything below the impressions
floor route straight to human review, never an automatic action.

**Decay/refresh insight, from this dataset's own label** (not borrowed from the paper): among
high-visibility pages (impressions_90d at or above the portfolio median), those stale 91+ days
show a higher decline rate than those refreshed in the last 30 days -- directionally consistent
with the paper's own freshness finding, though the effect here is more modest and I report the n
for every bucket rather than the flashiest ratio.

In [ ]:
FLOOR = 1000  # same volume floor as the Week-4 baseline rule

def reason_codes(row):
    reasons = []
    if row.decline_probability >= 0.6 and row.impressions_90d >= 500:
        reasons.append("model_flagged_high_confidence")
    if row.days_since_last_update >= 90 and row.impressions_90d >= FLOOR:
        reasons.append("stale_high_visibility")
    if 0 < row.avg_position <= 10 and row.decline_probability >= 0.5:
        reasons.append("page_one_at_risk")
    if row.impressions_90d < FLOOR:
        reasons.append("low_volume_low_confidence")
    if row.archetype == "small_edge_case_cluster":
        reasons.append("small_cluster_edge_case")
    return reasons or ["general_watch"]

def suggested_action(reasons):
    s = set(reasons)
    if "low_volume_low_confidence" in s or "small_cluster_edge_case" in s:
        return "human_review_only"
    if "stale_high_visibility" in s and "model_flagged_high_confidence" in s:
        return "refresh_priority"
    if "page_one_at_risk" in s:
        return "refresh_and_monitor"
    if "model_flagged_high_confidence" in s:
        return "review_candidate"
    return "monitor"

df["reason_codes"] = df.apply(reason_codes, axis=1)
df["reason_codes_str"] = df["reason_codes"].apply(lambda r: "|".join(r))
df["suggested_action"] = df["reason_codes"].apply(suggested_action)
df["rank"] = df["decline_probability"].rank(method="first", ascending=False).astype(int)

print(df["suggested_action"].value_counts())
print()
print(df.sort_values("rank").head(5)[["rank", "client_id", "decline_probability", "suggested_action",
                                        "reason_codes_str", "archetype"]].to_string(index=False))

# decay/refresh insight, grounded in our own label
df["freshness_bucket"] = pd.cut(df["days_since_last_update"], [-1, 30, 90, 180, 10**9],
                                 labels=["0-30", "31-90", "91-180", "181+"])
df["visibility_tier"] = np.where(df["impressions_90d"] >= df["impressions_90d"].median(),
                                  "high_visibility", "low_visibility")
decay = df.groupby(["visibility_tier", "freshness_bucket"], observed=True).agg(
    n=("content_id", "count"), decline_rate=("is_declining_label", "mean")).round(3)
print("\ndecay/refresh insight (n reported for every bucket):")
print(decay)

suggested_action
human_review_only      16503
monitor                 5838
refresh_and_monitor     3011
refresh_priority        2885
review_candidate        1763
Name: count, dtype: int64

 rank         client_id  decline_probability    suggested_action                               reason_codes_str                    archetype
    1 client_19581e27de             1.000000 refresh_and_monitor model_flagged_high_confidence|page_one_at_risk established_recently_updated
    2 client_7f2253d7e2             0.999921 refresh_and_monitor model_flagged_high_confidence|page_one_at_risk            young_high_volume
    3 client_7f2253d7e2             0.998358 refresh_and_monitor model_flagged_high_confidence|page_one_at_risk            young_high_volume
    4 client_7f2253d7e2             0.996519 refresh_and_monitor model_flagged_high_confidence|page_one_at_risk            young_high_volume
    5 client_7f2253d7e2             0.986168    review_candidate                  model_flagged_high_confi

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who this is for:** a content strategist or SEO lead deciding which pages to review first
this month -- a triage aid, not an autonomous system.

**What it is:** a ranked list of `decline_probability` (an *observed, measured* score from a
model *validated* out-of-sample at precision@20 = 0.75 on held-out clients) plus interpretable
reason codes and an archetype label, meant to shorten a manual review queue.

**Where it stops being valid:**
- Below the impressions floor (1,000/90d), the score is *not* validated -- Week 5's error
  analysis showed low-volume pages are where the model is most often confidently wrong in both
  directions.
- Trained and validated on this anonymized, teaching-scale sample (30,000 pages, 32 clients) --
  not FlyRank's production data, and not re-validated for any new client without repeating the
  Week-6 client-grouped check.
- `decline_probability` is *directional*, not causal: it says "this page's profile resembles
  pages that declined," never "refreshing this page will fix it" -- no experiment backs a causal
  claim here.
- One-time snapshot: the model was trained once, on one static export. It does not update itself
  and will drift (see section 4).

In [ ]:
pct_below_floor = (df["impressions_90d"] < FLOOR).mean()
print(f"share of the portfolio below the {FLOOR}-impression validation floor: {pct_below_floor:.1%}")
print(f"validated (held-out, client-grouped) numbers this tool is trusted by: {VALIDATED}")
print(f"Week-4 rule baseline, same held-out rows, for comparison: {BASELINE_VALIDATED}")

share of the portfolio below the 1000-impression validation floor: 55.0%
validated (held-out, client-grouped) numbers this tool is trusted by: {'precision@20': 0.75, 'precision@50': 0.8, 'precision@100': 0.74, 'roc_auc': 0.586, 'avg_precision': 0.589, 'base_rate': 0.511, 'split': 'client-grouped, 7 held-out clients'}
Week-4 rule baseline, same held-out rows, for comparison: {'precision@20': 0.5, 'precision@50': 0.56, 'precision@100': 0.54, 'roc_auc': 0.52}


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human review is required before any action, always** -- this is a triage list, not an
approval. Two checks a reviewer should run before batch-actioning any slice of the queue:

1. **Client concentration.** Week 6 found the model's false positives clustered inside a single
   client -- a sign it can partly learn "this client's baseline," not just page-level decline.
   The code below checks the same thing on this queue: if one client dominates a review batch,
   check whether the *client*, not the *page*, is driving the score before acting on all of them.
2. **Volume floor.** Anything under 1,000 impressions/90d is `human_review_only` by construction
   -- never auto-actioned, because the model's own validation doesn't cover that range reliably.

**Never automate:**
- Auto-publishing a refresh, rewrite, or content change based on `suggested_action` alone.
- Batch-actioning a review slice without checking client concentration first.
- Treating `small_edge_case_cluster` pages as a normal archetype -- there are too few of them
  (see the cluster sizes in section 1) to trust the archetype-level pattern; review individually.
- Reusing this exact model/thresholds for a brand-new client without re-running the Week-6
  grouped-holdout check on data that includes them.

In [ ]:
top50 = df.sort_values("rank").head(50)
top_client_share = top50["client_id"].value_counts(normalize=True).iloc[0]
top_client_name = top50["client_id"].value_counts().index[0]
print(f"top-50 queue: {top50['client_id'].nunique()} distinct clients, "
      f"largest single client is {top_client_share:.0%} of the batch")
if top_client_share > 0.30:
    print("-> exceeds the 30% concentration flag: check this client's page-level scores individually "
          "before actioning the batch, per the no-go list above.")

human_review_share = (df["suggested_action"] == "human_review_only").mean()
print(f"\n{human_review_share:.1%} of the portfolio routes to human_review_only "
      f"(below the volume floor or in the small edge-case cluster) -- never auto-actioned.")

top-50 queue: 6 distinct clients, largest single client is 74% of the batch
-> exceeds the 30% concentration flag: check this client's page-level scores individually before actioning the batch, per the no-go list above.

55.0% of the portfolio routes to human_review_only (below the volume floor or in the small edge-case cluster) -- never auto-actioned.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

What would tell me this playbook has gone stale, and needs a re-check or a retrain:

- **Base-rate drift.** If the portfolio's actual decline rate moves materially away from 0.511
  (the rate this model was validated against), the model's calibration is stale.
- **New client onboarded.** Week 6 showed client identity leaks into the score; scoring a client
  the model has never seen needs a fresh grouped-holdout check before its pages are trusted, not
  an assumption that 0.75 precision@20 carries over.
- **Concentration creep.** If the top-50 queue's single-client share (section 3) keeps climbing
  release over release, that's a sign the model is increasingly reading client identity rather
  than page signal -- a retrain-and-re-audit trigger on its own.
- **Cadence.** Absent any of the triggers above, re-validate on a rolling basis (e.g. quarterly)
  since this is a static, one-time fit with no online learning.

The code cell saves a small monitoring snapshot -- the numbers a future run would diff against
to check for the first two triggers automatically.

In [ ]:
monitoring_snapshot = {
    "base_rate": float(df["is_declining_label"].mean()),
    "top50_max_client_share": float(top_client_share),
    "pct_below_volume_floor": float(pct_below_floor),
    "n_pages": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "validated_reference": VALIDATED,
}
print(json.dumps(monitoring_snapshot, indent=2))

{
  "base_rate": 0.5420666666666667,
  "top50_max_client_share": 0.74,
  "pct_below_volume_floor": 0.5496,
  "n_pages": 30000,
  "n_clients": 32,
  "validated_reference": {
    "precision@20": 0.75,
    "precision@50": 0.8,
    "precision@100": 0.74,
    "roc_auc": 0.586,
    "avg_precision": 0.589,
    "base_rate": 0.511,
    "split": "client-grouped, 7 held-out clients"
  }
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exports for next week's paper. The ranked queue CSV regenerates from this notebook every
run (and is intentionally not committed -- the repo's leak-guard blocks data files under `work/`
by design); the metrics JSON and figures ARE committed, since they're the receipts the paper's
numbers need to trace back to.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue_cols = ["rank", "content_id", "client_id", "archetype", "decline_probability",
              "suggested_action", "reason_codes_str", "impressions_90d", "avg_position",
              "content_age_days", "days_since_last_update", "is_declining_label"]
df.sort_values("rank")[queue_cols].to_csv("work/outputs/w07_ranked_action_queue.csv", index=False)

metrics_payload = {
    "validated_model_metrics": VALIDATED,
    "validated_baseline_metrics": BASELINE_VALIDATED,
    "monitoring_snapshot": monitoring_snapshot,
    "action_counts": df["suggested_action"].value_counts().to_dict(),
    "archetype_names": archetype_names,
}
with open("work/outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics_payload, f, indent=2)

# Figure 1: validated precision@K, model vs baseline (the headline comparison for the paper)
ks = ["precision@20", "precision@50", "precision@100"]
fig, ax = plt.subplots(figsize=(6, 4))
width = 0.35
xpos = np.arange(len(ks))
ax.bar(xpos - width/2, [BASELINE_VALIDATED[k] for k in ks], width, label="Week-4 rule baseline")
ax.bar(xpos + width/2, [VALIDATED[k] for k in ks], width, label="Week-5/6 model")
ax.axhline(VALIDATED["base_rate"], color="gray", linestyle="--", linewidth=1, label="base rate")
ax.set_xticks(xpos); ax.set_xticklabels(ks)
ax.set_ylabel("precision (client-grouped holdout)")
ax.set_title("Validated precision@K: model vs baseline")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("work/figures/w07_precision_at_k.png", dpi=150)
plt.close(fig)

# Figure 2: decay/refresh insight
fig, ax = plt.subplots(figsize=(6, 4))
decay_plot = decay.reset_index()
for tier, sub in decay_plot.groupby("visibility_tier"):
    ax.plot(sub["freshness_bucket"].astype(str), sub["decline_rate"], marker="o", label=tier)
ax.set_ylabel("decline rate (is_declining_label)")
ax.set_xlabel("days since last update")
ax.set_title("Decay/refresh insight: decline rate by freshness x visibility")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("work/figures/w07_decay_refresh_insight.png", dpi=150)
plt.close(fig)

print("wrote work/outputs/w07_ranked_action_queue.csv "
      f"({len(df)} rows, not committed -- regenerated by this notebook)")
print("wrote work/outputs/w07_playbook_metrics.json (committed)")
print("wrote work/figures/w07_precision_at_k.png, work/figures/w07_decay_refresh_insight.png (committed)")


wrote work/outputs/w07_ranked_action_queue.csv (30000 rows, not committed -- regenerated by this notebook)
wrote work/outputs/w07_playbook_metrics.json (committed)
wrote work/figures/w07_precision_at_k.png, work/figures/w07_decay_refresh_insight.png (committed)
